# Notebook 3 (phase 2) — Contamination: why your good scores may be lying

Notebook 1 measured the system end to end on HotpotQA: the reference answer is given 57% of
the time. This notebook asks one more question of that number: **how much of it would the
model have scored with no system at all?**

It runs the **closed-book probe**: the same questions, the same generator, and no document.
Then it looks at the other side: a corpus built so that no model can have read it, and what
the same probe says there.

Everything is imported from `src/` and run as is. **Live cells** run on the fixtures with a
scripted generator (no key, no cost). **Archive cells** read the full runs of 2026-11-05
under `data/results/runs/contamination/`, the manifest of the in-house corpus and the
drafts of the golden set; the numbers in the prose are theirs. No database is needed.

## Part 1 — The problem

A language model is trained on a very large part of the public web. HotpotQA and SQuAD are
made of Wikipedia; QASPER is made of arXiv papers from 2017 to 2020; all three have been
public for years, questions and answers included. A model evaluated on them may have seen:

- **the source documents** (Wikipedia, the papers): it then knows many answers without
  being told;
- **the benchmark itself**: the questions next to their reference answers.

Either way the test data **contaminates** the evaluation: part of the score measures what
the model remembers, not what the system does. For a RAG the consequence is specific. The
whole point of retrieval is to give the model what it does not know. If the model already
knows the answer:

| Situation | What the score says | What is true |
|---|---|---|
| the search finds the passage, the model answers right | the system works | cannot tell: it might have answered anyway |
| the search misses the passage, the model answers right | the system works | the search failed and memory covered for it |
| a better search is tried, the score does not move | the improvement is useless | the questions never needed the search |

The third row is the expensive one: phase 3 will compare retrieval techniques, and a
benchmark the model answers from memory cannot tell a good one from a bad one.

Two words used below. **Closed book**: the question is asked with no document, like an exam
without notes. **Fresh**: published after the model existed, so that it cannot have been
read.

In [1]:
# --- Configuration: edit, then "Run All" ---
import os

from llm_testbench.config import find_repo_root

SAMPLE = os.environ.get("LLM_TESTBENCH_SAMPLE", "0") == "1"
ROOT = find_repo_root()  # the notebook runs from notebooks/phase-2/

# --- Experiment 1 (part 4.1): which archived runs to set side by side ------------------
# dataset label -> (closed-book arm, arm of the same questions on the gold context)
PAIRS = {
    "HotpotQA": ("contamination/hotpotqa-closed-book", "end-to-end/hotpotqa-gold"),
    "SQuAD 2.0": ("contamination/squad-closed-book", "generation-baseline/squad-gold-sonnet"),
    "QASPER": ("contamination/qasper-closed-book", "end-to-end/qasper-gold"),
}

# --- Experiment 2 (part 4.2): the end-to-end arm to take apart --------------------------
END_TO_END = "end-to-end/hotpotqa-retrieved"
CLOSED_BOOK = "contamination/hotpotqa-closed-book"
GOLD = "end-to-end/hotpotqa-gold"

# --- Experiment 3 (part 3.2): is the corpus fresh for this model? -----------------------
MODELS_TO_CHECK = [
    "anthropic/claude-sonnet-5-5",
    "openai/gpt-5.6-luna",
    "anthropic/claude-sonnet-6",
]

SHOW = 4  # how many example answers the archive cells print

In [2]:
from llm_testbench.config import Settings
from llm_testbench.eval.context import NoContext
from llm_testbench.eval.corpus.manifest import MANIFEST_PATH, Manifest
from llm_testbench.eval.generation import format_generation_run, load_generation_run
from llm_testbench.eval.golden.store import GOLDEN_DIR, read_drafts
from llm_testbench.eval.loaders.squad import SquadV2Loader
from llm_testbench.eval.suite import Suites, run_generation_arm
from llm_testbench.generation.answer import AnswerBuilder, is_refusal, parse_answer
from llm_testbench.llm import DataClass, ModelRef
from llm_testbench.llm.client import LLMClient
from llm_testbench.llm.prompts import PromptRegistry

fixtures = Settings(
    data_dir=ROOT / "data" / "cache", fixtures_dir=ROOT / "tests" / "fixtures", sample=True
)
RUNS = ROOT / "data" / "results" / "runs"


def run_of(name: str):
    return load_generation_run(RUNS / f"{name}.json")


def right(run) -> dict[str, bool]:
    """question id -> the reference answer is in the generated text (answerable ones)."""
    return {
        answer.example_id: bool(answer.metrics["contains_answer"])
        for answer in run.scored()
        if answer.answerable
    }

## Part 2 — The options

| How to know whether a benchmark is contaminated | What it gives | Limit | Verdict |
|---|---|---|---|
| **Compare dates**: the dataset's publication against the model's training cutoff | free | a cutoff is what the provider says; Wikipedia facts are older than any dataset built on them | necessary, proves nothing |
| **Closed-book probe**: ask the questions without any document | a number per dataset, per question | a low score does not prove the data is clean (part 5) | **built here** |
| **Completion probe**: give the start of a passage, ask for what follows, measure the overlap | evidence of verbatim memory | current models paraphrase or decline; noisy | not built (part 9) |
| **Fresh data**: evaluate on documents published after the model was released | contamination ruled out by construction | you have to build the corpus and write the questions | **built here** (notebook 4) |

The probe and the fresh corpus answer two different questions. The probe says how much of a
public benchmark's score is memory. The fresh corpus gives a place where the answer is
"none", so that what is measured there is the system.

## Part 3 — Guided reading

### 3.1 The probe is the generation runner with nothing in the context

Notebook 1 showed one runner fed by two kinds of context: the gold passages and the
retrieved chunks. The probe is a third kind, [`NoContext`](../../src/llm_testbench/eval/context.py#L77): it hands over no
evidence. Nothing else is special:

- an arm of [`config/eval_suites.toml`](../../config/eval_suites.toml) says
  `context = "none"` and `prompt = "closed_book"`; [`run_generation_arm`](../../src/llm_testbench/eval/suite.py#L425) builds the
  provider and the prompt from it;
- the prompt is a versioned file, [`prompts/closed_book.md`](../../prompts/closed_book.md):
  answer in one sentence from what you know, and if you do not know, answer exactly the
  sentinel sentence. Without that way out a model asked to answer will answer, and a guess
  cannot be told from a memory;
- the same questions as the other arms (same dataset, same size, same seed), so that each
  closed-book answer can be set next to the answer the same question got with documents;
- the same metrics: the reference answer is in the text or not, the sentinel was given or
  not. Citation metrics are computed too and mean nothing here (there is nothing to cite).

The live cell shows the request for one fixture question, then runs the closed-book arm of
the regression gate on the fixtures. The generator is scripted: it gives one fixed
sentence, so the rates below are those of that sentence, not of a model.

In [3]:
squad = SquadV2Loader(fixtures).load()
example = squad.examples[0]
given = await NoContext().context(example)
prompts = PromptRegistry.from_settings(fixtures)
request = AnswerBuilder(prompts, prompt_name="closed_book").build_request(
    example.question,
    list(given.evidence),
    model=ModelRef.parse("anthropic/claude-sonnet-5-5"),
    data_class=DataClass.PUBLIC,
)
print(f"evidence handed to the generator: {len(given.evidence)} excerpt(s)\n")
print("----- system prompt -----")
print(request.system)
print("----- user message -----")
print(request.turns[-1].content)

for raw in [
    "It rises in the Massif Central.",
    "I cannot answer this question from the provided documents.",
]:
    parsed = parse_answer(raw, [])
    print(f"\nanswer: {raw!r} -> says it does not know: {parsed.refused}")

gate = Suites.load(ROOT / "config" / "eval_suites.toml").get("gate")
arm = next(a for a in gate.generation if a.name == "squad-closed-book")
live = await run_generation_arm(
    "notebook", arm, settings=fixtures, client=LLMClient.from_settings(fixtures), ledger=None
)
print()
print(format_generation_run(live))

evidence handed to the generator: 0 excerpt(s)

----- system prompt -----
You answer a question from what you already know. No document is provided.

Rules:
1. Answer in one short sentence. Give the answer itself, not an explanation.
2. If you do not know, or if the question depends on a document you have not been given, answer exactly, and only: I cannot answer this question from the provided documents.
3. Do not guess. A wrong answer is worse than saying you do not know.
----- user message -----
Where does the Loire rise?

answer: 'It rises in the Massif Central.' -> says it does not know: False

answer: 'I cannot answer this question from the provided documents.' -> says it does not know: True

squad-closed-book — squad_v2, 4 questions, generator anthropic/claude-sonnet-5-5, context: none (closed book)  [NOT PUBLISHABLE: sample mode]
metric                               rate     95% interval     n
answer F1 (SQuAD)                   0.000   [0.000, 0.000]     2
cites an excerpt (ans

### 3.2 Fresh by construction: a release date, not a cutoff

The usual sentence is "published after the model's training cutoff". A cutoff is a
statement by a provider, sometimes approximate, sometimes absent. A **release date** is a
fact, and it is enough: a model cannot have been trained on a text first published after it
was available.

[`config/corpus.toml`](../../config/corpus.toml) lists each model the testbench uses with a
day on which it is known to have been available and where that day comes from. The
freshness floor has to be after all of them, or [`CorpusConfig`](../../src/llm_testbench/eval/corpus/manifest.py#L59) refuses to
load the file. Every document of the corpus was **first** submitted on or after the floor
(the first version: a 2013 paper revised yesterday is not fresh). The manifest keeps the
floor and the models, and [`Manifest.fresh_for`](../../src/llm_testbench/eval/corpus/manifest.py#L146) answers for one model:

- a model the manifest lists: fresh;
- a model it does not list: **not fresh**. The property is established, never assumed: the
  day the testbench moves to a newer model, the corpus is no longer known to be fresh for
  it, and the code says so.

In [4]:
manifest = Manifest.load(ROOT / MANIFEST_PATH)
submitted = sorted(entry.submitted.date() for entry in manifest.documents)
print(
    f"{manifest.name}: {len(manifest.documents)} documents, first submitted between "
    f"{submitted[0]} and {submitted[-1]}"
)
print(f"freshness floor: {manifest.freshness_floor}\n")
print(f"{'model':<22} {'known available by':<20} evidence")
for name, model in manifest.models.items():
    print(f"{name:<22} {model.available_by!s:<20} {model.evidence}")
print()
for model in MODELS_TO_CHECK:
    print(f"fresh for {model:<30} {manifest.fresh_for(model)}")

arxiv-fresh: 200 documents, first submitted between 2026-10-20 and 2026-10-21
freshness floor: 2026-10-20

model                  known available by   evidence
claude-sonnet-5-5      2026-10-19           docs/decisions.md, 2026-10-19: default generator, notebooks 1 and 2 re-executed with it
gpt-5.6-luna           2026-09-03           config/llm_pricing.toml: OpenAI pricing page recorded on 2026-09-03
gpt-5.6-terra          2026-09-03           config/llm_pricing.toml: OpenAI pricing page recorded on 2026-09-03

fresh for anthropic/claude-sonnet-5-5    True
fresh for openai/gpt-5.6-luna            True
fresh for anthropic/claude-sonnet-6      False


## Part 4 — Run

### 4.1 Three public datasets, no document

The archive cell sets, for each dataset, the closed-book run next to the run of the same
questions on the gold context (notebook 1). "Reference answer given" is computed on the
answerable questions; "says it does not know" is the share of those where the model gave
the sentinel.

In [5]:
print(
    f"{'dataset':<10} {'questions':>9}   {'closed book: reference answer given':<36} "
    f"{'says it does not know':>22}   {'gold context':>12}"
)
for label, (closed_name, gold_name) in PAIRS.items():
    closed, gold = run_of(closed_name), run_of(gold_name)
    summary = closed.metrics["contains_answer"]
    unknown = closed.metrics["over_refusal"]
    interval = f"{summary.mean:.2f} [{summary.low:.2f}, {summary.high:.2f}]"
    print(
        f"{label:<10} {summary.examples:>9}   {interval:<36} {unknown.mean:>22.2f}   "
        f"{gold.metrics['contains_answer'].mean:>12.2f}"
    )
    assert closed.metrics["truncated"].mean == 0.0, "a cut answer is not an answer (part 5)"

print(f"\n{SHOW} HotpotQA answers given without any document, marked right:")
shown = [a for a in run_of(CLOSED_BOOK).scored() if a.metrics["contains_answer"]][:SHOW]
for answer in shown:
    print(f"  - {answer.answer[:170]}")

dataset    questions   closed book: reference answer given   says it does not know   gold context
HotpotQA         100   0.53 [0.43, 0.63]                                      0.20           0.85
SQuAD 2.0         50   0.38 [0.26, 0.52]                                      0.34           0.92
QASPER            50   0.00 [0.00, 0.00]                                      0.98           0.28

4 HotpotQA answers given without any document, marked right:
  - Shirley Temple, who played Corliss Archer in the 1945 film *Kiss and Tell*, later served as U.S. Ambassador to Ghana and Czechoslovakia, and as Chief of Protocol of the U
  - The 1925 Birthday Honours were made by King George V, who was born in 1865.

**1865**
  - Johnny Galecki appeared in *Suicide Kings* (1997).
  - Michael Strahan spent his entire 15-year NFL career with the New York Giants.


**HotpotQA: with no document at all, the generator gives the reference answer on 53% of the
questions** (between 43% and 63%). These are "multi-hop" questions written to need two
Wikipedia paragraphs; the model has read Wikipedia. Read the answers printed above: it
performs both hops from memory.

**SQuAD 2.0: 38%** of the answerable questions, and it says it does not know on a third.
SQuAD questions are tied to one paragraph ("what did the report conclude?"), which protects
them a little.

**QASPER: 0%, and "I do not know" 98% of the time.** That is not a clean bill of health,
and part 5 comes back to it: "which baselines do they compare with?" cannot be answered
without knowing which paper is meant. The papers themselves are from 2017 to 2020 and well
inside any training set.

### 4.2 What is left of an end-to-end score

Notebook 1 reported 57% end to end on HotpotQA. The same 100 questions were asked closed
book. Crossing the two, question by question, splits the 57 right answers into those the
model also gives alone and those that needed the system.

In [6]:
closed, system, gold = right(run_of(CLOSED_BOOK)), right(run_of(END_TO_END)), right(run_of(GOLD))
ids = sorted(set(closed) & set(system) & set(gold))
known = [i for i in ids if closed[i]]
unknown = [i for i in ids if not closed[i]]

print(
    f"{len(ids)} questions — right answers: closed book {len(known)}, "
    f"end to end {sum(system[i] for i in ids)}, gold context {sum(gold[i] for i in ids)}\n"
)
print(f"{'':<44} {'end to end right':>17} {'gold context right':>19}")
for label, group in [
    ("the model knows the answer alone", known),
    ("the model does not know it", unknown),
]:
    print(
        f"{label + ' (' + str(len(group)) + ')':<44} "
        f"{sum(system[i] for i in group):>10} ({sum(system[i] for i in group) / len(group):.0%}) "
        f"{sum(gold[i] for i in group):>12} ({sum(gold[i] for i in group) / len(group):.0%})"
    )

both = sum(system[i] for i in known)
print(
    f"\nof the {sum(system[i] for i in ids)} right end-to-end answers, {both} are on questions "
    f"the model answers without any document"
)
print(
    f"questions the model knew alone and the system got wrong or refused: "
    f"{sum(not system[i] for i in known)}"
)

100 questions — right answers: closed book 53, end to end 57, gold context 85

                                              end to end right  gold context right
the model knows the answer alone (53)                38 (72%)           49 (92%)
the model does not know it (47)                      19 (40%)           36 (77%)

of the 57 right end-to-end answers, 38 are on questions the model answers without any document
questions the model knew alone and the system got wrong or refused: 15


Three readings of the same 100 questions.

**Two thirds of the system's right answers are on questions the model answers alone**: 38 of
57. The headline score is mostly memory.

**The system's own contribution is measured on the questions the model does not know**: 47
of them. There the system is right 40% of the time (19 of 47), where the gold context gets
77% (36 of 47). That gap, 40 against 77, is what retrieval has to close in phase 3, and it
is twice as wide as the gap of the headline numbers (57 against 85). A benchmark made of
known answers flatters the baseline and shrinks the room for improvement at the same time.

**On 15 questions the model knew the answer and the system did not give it.** The prompt of
the system says to rely only on the excerpts and to refuse otherwise; when a paragraph was
missing, the model refused although it knew. That is the behaviour asked for, and the right
one for documents the model has not read. On Wikipedia it reads as the RAG being worse than
no RAG: 57% against 53%, barely better than doing nothing.

None of this says the system is bad. It says HotpotQA cannot tell.

### 4.3 The same probe on a corpus no model has read

The in-house corpus (notebook 4) is 200 arXiv papers first submitted on 2026-10-20 or
later, after every model used here was released. 83 questions were drafted on 30 of them.
Before any review, each draft was asked to the generator closed book
([`probe_all`](../../src/llm_testbench/eval/golden/cli.py#L112)); the answers are stored with the drafts.

In [7]:
drafts = read_drafts(ROOT / GOLDEN_DIR)
probed = [draft for draft in drafts if draft.closed_book_answer is not None]
answerable = [draft for draft in probed if draft.answerable]
said_unknown = [draft for draft in probed if is_refusal(draft.closed_book_answer)]
guessed = [draft for draft in answerable if draft.closed_book_correct]

print(f"{len(probed)} drafts asked without any document ({len(answerable)} of them have an answer)")
print(f"  said it did not know : {len(said_unknown)}")
print(f"  gave an answer       : {len(probed) - len(said_unknown)}")
print(
    f"  gave the reference answer: {len(guessed)} of {len(answerable)} "
    f"({len(guessed) / len(answerable):.0%})\n"
)
for draft in guessed:
    print(f"[{draft.category.value}] {draft.question}")
    print(f"   reference  : {draft.answer}")
    print(f"   closed book: {draft.closed_book_answer}\n")

83 drafts asked without any document (68 of them have an answer)
  said it did not know : 78
  gave an answer       : 5
  gave the reference answer: 2 of 68 (3%)

[temporal] In AGGR-Mamba final-stage optimization, how many iterations elapse between the end of the 100-iteration warmup period and the first validation performed at the 500-iteration interval?
   reference  : 400 iterations.
   closed book: 400 iterations (warmup ends at iteration 100 and the first validation is at iteration 500).

[temporal] Given TF-IG's 30-second PPG windows and 2-second shift between adjacent frames, how much temporal overlap is there between consecutive windows?
   reference  : 28 seconds.
   closed book: Consecutive windows overlap by 28 seconds (about 93%), since 30 s − 2 s = 28 s.



On the fresh corpus the generator says it does not know 78 times out of 83 and gives the
reference answer on **2 of the 68 drafts that have one (3%)**, against 53% on HotpotQA.
Both are printed above, and neither is memory: each is arithmetic on numbers the question
itself states (500 minus 100, 30 minus 2). They are **guessable questions**, and that is
the second use of the probe: every draft is flagged before its review, and a question the
model answers alone is rewritten or rejected (notebook 4).

These drafts are not reviewed yet. The comparison that will count is the golden set after
review; the order of magnitude will not change.

## Part 5 — Pitfalls

**An empty answer is not an answer, and the first run of this probe said it was.** The
generator reasons before it writes; those tokens are output tokens, counted against the
output cap and absent from the text. The probe asks for one sentence, so it was capped at
120 tokens. The reasoning used them up: **56 answers in 100 came back empty**, with a
finish reason of `length` and no error. An empty text is not the sentinel, so the run
reported that the model "answers 99% of the questions and is right on 31%". The real
figures are 80% and 53%. The cap is now the API's own (4,096), every answer records whether
it was cut, and the archive cell above asserts that none was. A probe that measures
"does the model answer" must be able to tell an answer from nothing.

**A low closed-book score does not prove a clean dataset.** QASPER scores 0%: its questions
name no paper. The model may know every one of those papers; the probe cannot ask. Closed
book finds contamination when it is there and the question stands alone; it does not rule
it out.

**"Unanswerable" in SQuAD means "not in this paragraph", not "unknowable".** Closed book,
the model answers 30% of them, some correctly from memory. The correct-refusal rate has no
meaning without the paragraph; only the answerable half is read here.

**"The reference answer is in the text" is lenient on comparisons.** "Which airport is
busier, A or B?" has the reference "B"; an answer saying "A is busier than B" contains it.
The rate is an upper bound on what the model knows; a judge would do better (notebook 2).

**Telling the model to say "I do not know" changes what is measured.** Without the way
out, a guess and a memory look the same. With it, a cautious model under-reports what it
knows: on HotpotQA it says it does not know on 20% of the questions. The probe measures
what the model is willing to assert, which is what matters for a system that must not
invent.

**Fresh is a dated property.** The corpus is fresh for the models listed in its manifest.
The day a newer generator is plugged in, [`Manifest.fresh_for`](../../src/llm_testbench/eval/corpus/manifest.py#L146) answers false,
and the corpus has to be rebuilt with a later floor before its numbers mean "no memory".
And "first submitted to arXiv on a given day" is not "never seen before": a paper can have
circulated earlier.

## Part 6 — Guided experiments

1. **Another split of the same table** (part 4.2). Set `GOLD = "end-to-end/hotpotqa-retrieved"`
   and `END_TO_END = "end-to-end/hotpotqa-gold"`: the roles swap, and the column on the
   right now reads what the search costs on each group.
2. **Is the corpus fresh for another model?** Add a model name to `MODELS_TO_CHECK`. Then
   open [`config/corpus.toml`](../../config/corpus.toml) and read what would have to be
   added, and what happens when its date is after the floor.
3. **Your own questions, closed book.** `make eval SUITE=contamination ARMS=hotpotqa-closed-book`
   reruns one arm (about $0.30: most of it is the model's reasoning). To probe another
   size or another seed, change `max_examples` or `seed` of the arm in
   [`config/eval_suites.toml`](../../config/eval_suites.toml): the questions of the paired
   arms must change with it, or the cross-table of part 4.2 has nothing to cross.

## Part 7 — The numbers

Closed book, `claude-sonnet-5-5`, runs of 2026-11-05 (`data/results/runs/contamination/`),
95% bootstrap intervals:

| Dataset | Questions with an answer | Reference answer given, no document | Says it does not know | Same questions, gold context |
|---|---|---|---|---|
| HotpotQA (pool of 1,000) | 100 | **0.53** [0.43, 0.63] | 0.20 | 0.85 |
| SQuAD 2.0 | 50 | **0.38** [0.26, 0.52] | 0.34 | 0.92 |
| QASPER | 50 | 0.00 | 0.98 | 0.28 (metric not usable there) |
| In-house drafts (fresh arXiv, unreviewed) | 68 | **0.03** (2 of 68) | 0.94 of the 83 drafts | to come, after the review |

HotpotQA, the 100 questions crossed (part 4.2): 57 right end to end, of which 38 on
questions the model answers alone; on the 47 it does not know, 40% end to end against 77%
on the gold context.

No published score exists for these probes on these models. Published contamination
studies use other protocols and other models; nothing here is compared with them.

Cost: $0.62 for the three public probes (about $0.002 to $0.003 a question, nearly all of
it reasoning tokens) and $0.07 for the 83 drafts.

## Part 8 — Interview questions

<details>
<summary>Five questions, with the answers expected (click to open)</summary>

**1. Your RAG scores 57% on a public multi-hop benchmark. What do you check before
believing it?**
What the model scores on the same questions without retrieval. Here: 53%. Then the cross
table: of the 57 right answers, 38 are on questions the model answers alone. The system's
own contribution is its score on the questions the model does not know: 40%.

**2. What is contamination, concretely, for a RAG evaluation?**
The model has seen the test documents, or the questions with their answers, during
training. For a RAG it means the generator can answer without the retrieved context, so the
evaluation cannot tell a working retrieval from a broken one, and improvements to retrieval
do not show.

**3. How do you get evaluation data a model has not seen?**
By date, and by a date that is a fact: documents first published after the model was
released. Record the floor and the models it holds for; re-check when the model changes.
Then write the questions yourself (a small golden set), and ask them closed book to verify
that none can be guessed.

**4. Your closed-book probe scores 0% on a dataset. Is the dataset clean?**
Not necessarily. Look at the questions: if they do not stand alone ("what do they compare
with?"), the model cannot answer whatever it knows. A closed-book probe detects
contamination when the question is self-contained; a zero is only evidence when the
questions are.

**5. Your probe reports that the model answers 99% of the questions and is right on 31%.
What do you look at first?**
The raw answers. In this project 56 of 100 were empty strings: the model had spent its
output budget reasoning and the cap cut it before it wrote anything. Empty is not "I do not
know", so it counted as an answer. Always record the finish reason, and never trust a rate
before reading ten rows of what it was computed on.

</details>

## Part 9 — What we did not do, and why

- **No completion probe.** Giving the start of a passage and measuring how much of the
  continuation comes back word for word is the classic test of verbatim memory. Current
  models paraphrase or decline to continue copyrighted-looking text, so the signal is weak,
  and the closed-book probe had already answered the question for a quarter of the price.
- **No probe of the benchmark itself** (asking for a question's reference answer given its
  identifier, or for the next question of the file). It separates "read Wikipedia" from
  "read the benchmark"; for a RAG evaluation the consequence is the same.
- **No judge on the closed-book answers.** "The reference answer is in the text" is an
  upper bound (part 5). The calibrated judge of notebook 2 will grade them with the rest of
  the judged baseline.
- **No other model.** One generator, the system's. A probe per model is one line of
  configuration and about thirty cents.
- **No measurement on the reviewed golden set yet.** The 83 drafts wait for their review;
  the 3% above is on drafts.

## Glossary

| Term | Meaning |
|---|---|
| **contamination** | the evaluation data, or its source, was in the model's training data |
| **memorization** | the model reproduces or uses what it saw in training, without being given it |
| **closed book** | the question asked with no document; what the model answers alone |
| **closed-book probe** | running an evaluation set closed book to see how much of its score needs no system |
| **sentinel sentence** | the exact sentence the model must give when it does not know; recognized by code |
| **training cutoff** | the date after which the model's training data stops, as stated by its provider |
| **release date** | the day the model became available: a fact, and later than its training data |
| **freshness floor** | the date on or after which every document of the corpus was first published |
| **fresh for a model** | first published after that model was released |
| **guessable question** | one that can be answered from the question itself or from general knowledge |
| **output cap** | the maximum number of tokens a call may generate, reasoning included |
| **cut answer** | an answer that stopped on the output cap: incomplete, possibly empty |